In [0]:
Problem_statement = """
 Employee:
 +----+-------+--------+--------------+
 | Id | Name  | Salary | DepartmentId |
 +----+-------+--------+--------------+
 | 1  | Joe   | 70000  | 1            |
 | 2  | Jim   | 90000  | 1            |
 | 3  | Henry | 80000  | 2            |
 | 4  | Sam   | 60000  | 2            |
 | 5  | Max   | 90000  | 1            |
 +----+-------+--------+--------------+

Department:
 +----+----------+
 | Id | Name     |
 +----+----------+
 | 1  | IT       |
 | 2  | Sales    |
 +----+----------+

find employees who have the highest salary in each of the departments.
 +------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Jim      | 90000  |
| Sales      | Henry    | 80000  |
+------------+----------+--------+

Explanation:

Max and Jim both have the highest salary in the IT department and Henry has the highest salary in the Sales department.
"""


employee_data = [
    (1, "Joe", 70000, 1),
    (2, "Jim", 90000, 1),
    (3, "Henry", 80000, 2),
    (4, "Sam", 60000, 2),
    (5, "Max", 90000, 1),
]
employee_df = spark.createDataFrame(employee_data, schema=["Id", "Name", "Salary", "DepartmentId"])

department_data = [
    (1, "IT"),
    (2, "Sales"),
]
department_df = spark.createDataFrame(department_data, schema=["Id", "Name"])

display(employee_df)
display(department_df)



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

In [0]:
win_spec = Window.partitionBy('DepartmentId').orderBy(col('Salary').desc())
hs = employee_df.withColumn('rk', dense_rank().over(win_spec)).filter(col('rk')==1).drop('rk')
hs.display()

In [0]:
hs.join(department_df.alias('d'), hs.DepartmentId==col('d.id'), 'inner')\
    .select(col('d.Name').alias('Department'),
            hs.Name.alias('Employee'),
            hs.salary.alias('Salary')).show()